# Phase 3 — Purpose-Driven Exploratory Data Analysis (EDA)
**Project**: InsightPath — SAS CU Hackathon Round 2 Analytics  
**Module**: `notebooks/03_eda/03_eda.ipynb`  
**Analytical Scope**: Research Questions RQ1–RQ9

---

### Objectives of Phase 3
Phase 3 establishes the empirical, distributional, and structural foundations of the four data science talent datasets prior to formal hypothesis testing (Phase 4) and predictive modeling (Phases 5–6):
1. **Market Structure (RQ1)**: Role hiring volume and employer concentration across 1,602 firm requisitions.
2. **Compensation & Experience (RQ2)**: Experience elasticity slopes, salary distributions, and bargaining spreads.
3. **Geographic Distribution (RQ3)**: Vacancy share and wage bracket composition across 7 macro clusters.
4. **Skill Ecosystem (RQ4)**: Prevalence rankings of foundational vs specialized skills across 15,841 postings.
5. **Skill-to-Salary Associations (RQ5)**: High-salary differentials ($\ge 15	ext{L}$) for top technical skills without target leakage.
6. **Junior Competency Differentiation (RQ6)**: Technical skill distributions and salary-hike separation ($N = 139$).
7. **Senior Behavioral Profiling (RQ7)**: Big Five personality traits across client consulting success classes ($N = 161$).
8. **Model-Readiness Diagnostics (RQ8)**: Class balance, distribution skewness, collinearity, and leakage safeguards.
9. **Descriptive Synthesis (RQ9)**: Conceptual triangulation across market, junior, and senior lenses without row-level joins.


## 1. Environment Configuration & Module Ingestion


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root to path
sys.path.append(os.path.abspath("../.."))

from src.analysis.eda_summary import generate_all_eda_tables
from src.visualization.figure_generators import generate_all_eda_figures
from src.visualization.style import apply_publication_theme

apply_publication_theme()
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 1000)
print("Environment and analytical modules initialized successfully.")


## 2. Ingestion of Cleaned Analytical Datasets (Phase 2 Inputs)


In [ ]:
ds_path = "../../data/processed/data_science_jobs_processed.csv"
aj_path = "../../data/processed/analytics_jobs_processed.csv"
jds_path = "../../data/processed/jds_processed.csv"
jds_sens_path = "../../data/processed/jds_sensitivity_3291_removed.csv"
sds_path = "../../data/processed/sds_processed.csv"

df_ds = pd.read_csv(ds_path)
df_aj = pd.read_csv(aj_path)
df_jds = pd.read_csv(jds_path)
df_jds_sens = pd.read_csv(jds_sens_path)
df_sds = pd.read_csv(sds_path)

print(f"DataScience Jobs:    {len(df_ds):,} rows x {len(df_ds.columns)} cols")
print(f"Analytics Jobs:      {len(df_aj):,} rows x {len(df_aj.columns)} cols")
print(f"JDS Baseline:        {len(df_jds)} rows x {len(df_jds.columns)} cols")
print(f"JDS Sensitivity:     {len(df_jds_sens)} rows (ID 3291 removed)")
print(f"SDS Personality:     {len(df_sds)} rows x {len(df_sds.columns)} cols")


## 3. Schema & Integrity Validation


In [ ]:
assert len(df_ds) == 1602, "Row count mismatch in DataScience Jobs"
assert len(df_aj) == 15841, "Row count mismatch in Analytics Jobs"
assert len(df_jds) == 139, "Row count mismatch in JDS Baseline"
assert len(df_sds) == 161, "Row count mismatch in SDS"
assert set(df_jds["salary_hike_high_or_low"].unique()) == {0, 1}, "Invalid JDS target"
assert set(df_sds["success_classification_high_low"].unique()) == {0, 1}, "Invalid SDS target"
assert set(df_aj["is_high_salary"].unique()) == {0, 1}, "Invalid Analytics high salary indicator"

print("Integrity assertions: PASS (All sample sizes and target domains verified).")


## 4. Execution of EDA Pipeline (Tables & Publication Figures)


In [ ]:
tables_dir = "../../outputs/tables/phase3"
figs_dir = "../../outputs/figures/phase3"

# Generate 13 summary tables
tables = generate_all_eda_tables(df_ds, df_aj, df_jds, df_sds, output_dir=tables_dir)

# Generate 15 publication figures
figures = generate_all_eda_figures(df_ds, df_aj, df_jds, df_sds, output_dir=figs_dir)

print(f"Generated {len(tables)} analytical tables in {tables_dir}")
print(f"Generated {len(figures)} publication figures in {figs_dir}")


## 5. RQ1: Macro Market Structure & Role Demand
**Research Question**: Which Data Science job roles and hiring organizations show the highest observed job demand?
**Dataset**: `DataScience Jobs` ($N = 1,602$)


In [ ]:
df_roles = pd.read_csv(os.path.join(tables_dir, "phase3_role_demand.csv"))
print("=== Role Demand Summary Table ===")
display(df_roles[["demand_rank", "job_title", "total_openings", "postings_count", "pct_of_total_openings", "mean_avg_salary_lakh", "mean_min_experience"]])

df_top_comp = pd.read_csv(os.path.join(tables_dir, "phase3_company_demand.csv"))
print("\n=== Top 10 Hiring Organizations ===")
display(df_top_comp[["company_rank", "company_name", "total_openings", "postings_count", "pct_of_market_openings", "cumulative_pct_openings"]].head(10))


## 6. RQ2: Compensation & Experience Elasticity
**Research Question**: How does required experience relate to advertised salary?
**Datasets**: `DataScience Jobs` ($N = 1,602$) & `Analytics Jobs` ($N = 15,841$)


In [ ]:
df_exp = pd.read_csv(os.path.join(tables_dir, "phase3_experience_summary.csv"))
print("=== Experience Distributions & Salary Correlations ===")
display(df_exp)

df_sal = pd.read_csv(os.path.join(tables_dir, "phase3_salary_summary.csv"))
print("\n=== Compensation Summary by Experience Tier ===")
display(df_sal[df_sal["segment_type"] == "Experience Tier"])


## 7. RQ3: Regional Geographic Analytics Demand
**Research Question**: Which locations demonstrate the greatest observed analytics job demand?
**Dataset**: `Analytics Jobs` ($N = 15,841$)


In [ ]:
df_loc = pd.read_csv(os.path.join(tables_dir, "phase3_location_summary.csv"))
print("=== Geographic Distribution & High-Salary Share Across 7 Macro Clusters ===")
display(df_loc)


## 8. RQ4: Technical Skill Demand Landscape
**Research Question**: Which technical skills appear most frequently in Analytics job postings?
**Dataset**: `Analytics Jobs` ($N = 15,841$)


In [ ]:
df_skills = pd.read_csv(os.path.join(tables_dir, "phase3_skill_frequency.csv"))
print("=== Top 20 In-Demand Analytics Skills ===")
display(df_skills.head(20))


## 9. RQ5: Skills Associated with Premium Compensation ($\ge 15	ext{L}$)
**Research Question**: Which skills are associated with higher observed salary levels?
**Dataset**: `Analytics Jobs` ($N = 15,841$)  
*Safeguard*: Zero target leakage (`salary_rank` and `salary_midpoint` excluded).


In [ ]:
df_skill_sal = pd.read_csv(os.path.join(tables_dir, "phase3_skill_salary_comparison.csv"))
print("=== Skills Ranked by High-Salary Prevalence Differential ===")
display(df_skill_sal.head(15))


## 10. RQ6: Junior Data Scientist Technical Competencies & Hike Differentiation
**Research Question**: Which technical skill dimensions associate with high salary-hike classification?
**Dataset**: `JDS Skill Traits` ($N = 139$ Baseline, $N = 137$ Sensitivity)


In [ ]:
df_jds_tbl = pd.read_csv(os.path.join(tables_dir, "phase3_jds_summary.csv"))
print("=== JDS Technical Skill Metrics & Class Divergence (N = 139) ===")
display(df_jds_tbl[["skill_dimension", "overall_mean", "low_hike_mean", "high_hike_mean", "mean_difference", "cohens_d_effect_size", "exploratory_p_value"]])


## 11. RQ7: Senior Data Scientist Personality Profiles & Consulting Success
**Research Question**: Which Big Five personality dimensions associate with consulting success?
**Dataset**: `SDS Personality Traits` ($N = 161$)


In [ ]:
df_sds_tbl = pd.read_csv(os.path.join(tables_dir, "phase3_sds_summary.csv"))
print("=== SDS Personality Trait Metrics & Class Divergence (N = 161) ===")
display(df_sds_tbl[["personality_trait", "overall_mean", "low_success_mean", "high_success_mean", "mean_difference", "cohens_d_effect_size", "exploratory_p_value"]])


## 12. RQ8: Exploratory Model-Readiness Diagnostics
**Research Question**: What structural, distributional, and leakage considerations govern downstream modeling?
**Scope**: All 4 Datasets (Diagnostic Screening Only; Zero ML Models Fitted)


In [ ]:
df_diag = pd.read_csv(os.path.join(tables_dir, "phase3_diagnostic_summary.csv"))
print("=== Model-Readiness Diagnostic Audit ===")
display(df_diag[["dataset", "feature", "diagnostic_type", "observed_pattern", "implication"]])


## 13. RQ9: Descriptive Cross-Dataset Synthesis
**Research Question**: How do empirical findings connect conceptually across the talent lifecycle?
**Rule**: Conceptual Triangulation Only; Zero Row-Level Merging.


In [ ]:
df_rq = pd.read_csv(os.path.join(tables_dir, "phase3_rq_figure_map.csv"))
print("=== Complete Research Question Coverage Map ===")
display(df_rq[["research_question", "datasets_used", "primary_figures", "key_exploratory_finding"]])
